# 04 — Portfolio strategies and backtest (100 experiments)

In this notebook we compare the strategies of part 2 of the project brief:

| | Strategy | Idea |
|---|---|---|
| a) | **Equally weighted (EW)** | the same weight in every stock (main benchmark) |
| b) | **Markowitz mean-variance** | maximise return − (λ/2)·variance, with risk aversion λ = 3 |
| c) | **Global minimum variance (GMV)** | the portfolio with the lowest possible volatility |
| d) | **Max Sharpe** | the portfolio with the highest expected Sharpe ratio |
| e) | **Inverse volatility** and **Risk parity (ERC)** | weights inversely proportional to volatility / equal risk contributions |
| f) | **Most diversified** and **Max decorrelation** | maximise the diversification ratio / minimise the average correlation |

There are 8 strategies. **SPY** (the investable S&P 500 ETF) is used as the market benchmark.

### Conventions (the same for every strategy)

| Item | Choice |
|---|---|
| Universe | 23 S&P 500 stocks chosen in notebook `01` |
| Data | adjusted close prices (Yahoo Finance), daily, 2010-01-04 to 2026-09-18 |
| Returns | **simple** returns for the portfolios (log returns were only used for the stylised facts) |
| Experiments | **100**, random seed **2026** |
| Each experiment | one **contiguous 3-year period** (756 trading days) + **12 stocks** drawn from the 23 |
| Split | years 1–2 = estimation; **year 3 = out-of-sample evaluation** (252 days) |
| Estimation window | rolling, **1 year** (252 days), using only data from before the rebalancing date |
| Rebalancing | **semi-annual** (126 trading days); between dates the weights **drift** with prices |
| Constraints | long-only (weights ≥ 0), fully invested (sum = 1), no leverage |
| Risk-free rate | **3-year** Treasury yield (FRED, series `DGS3`); daily rate = annual rate / 252 |
| Transaction costs | **10 bps** on the traded value (Σ\|Δw\|); robustness checks with 0, 25 and 50 bps |
| Benchmarks | EW (main) and SPY buy & hold |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')   # matplotlib's default style (DejaVu Sans font) -> the same charts on every computer
plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 250)

os.makedirs('../figures', exist_ok=True)
os.makedirs('../results', exist_ok=True)

## 1. Backtest parameters

All the parameters are here at the top, so that it is easy to change them and run everything again.

In [ ]:
SEED = 2026                 # random seed -> guarantees that the 100 experiments are always the same
N_EXPERIMENTS = 100         # number of 3-year datasets
N_ASSETS = 12               # number of stocks drawn in each experiment (out of 23)

DAYS_PER_YEAR = 252                     # trading days in one year
DATASET_DAYS = 3 * DAYS_PER_YEAR        # 756 days = 3 years
EVAL_START = 2 * DAYS_PER_YEAR          # day 504: the first 2 years are used for estimation
ESTIMATION_WINDOW = DAYS_PER_YEAR       # rolling 1-year window to estimate means and covariances
REBALANCE_INTERVAL = 126                # semi-annual (126 trading days = 6 months)

COST = 0.0010               # 10 bps (0.10%) per unit traded
RISK_AVERSION = 3           # Markowitz lambda

## 2. Data: prices, simple returns and risk-free rate

- We use **simple returns** $R_t = P_t / P_{t-1} - 1$ because the return of a portfolio is the weighted average of the simple returns of the assets
  (with log returns this is not true).
- Adjusted prices already include dividends and *splits*. All the stocks are in USD and listed on US exchanges, so there is no currency conversion
  and the calendar is the same (we used the SPY calendar in notebook `01`).
- **SPY vs ^GSPC:** ^GSPC is the S&P 500 index — it is not investable and its price does not include dividends. SPY is an ETF that can be bought
  (it has management fees of ~0.09%/year) and its adjusted price includes dividends. That is why the market benchmark in the backtest is **SPY**.

In [ ]:
prices = pd.read_csv('data/clean_prices.csv', index_col=0, parse_dates=True)

# The 23 stocks = every column except SPY and the ^GSPC index
stocks = [column for column in prices.columns if column not in ['SPY', '^GSPC']]
print('Number of stocks in the universe:', len(stocks))
print(stocks)

# Daily simple returns (the first row is NaN because there is no previous price -> we drop it)
returns = prices.pct_change().iloc[1:]

print('\nPeriod:', returns.index.min().date(), 'to', returns.index.max().date(), '|', len(returns), 'trading days')
print('Missing values per stock (only TSLA, which went public in June 2010):')
print(returns[stocks].isna().sum()[returns[stocks].isna().sum() > 0])

### Universe summary (full period)

Before comparing strategies, we look at the characteristics of each stock: annualised return and volatility, maximum drawdown
and average correlation with the other stocks. Low correlations = more room to diversify.

In [ ]:
# GICS sector of each stock (the same as in notebook 01)
sector = {
    'NVDA': 'Information Technology', 'AAPL': 'Information Technology', 'MSFT': 'Information Technology',
    'AVGO': 'Information Technology', 'AMD': 'Information Technology', 'MU': 'Information Technology',
    'BRK-B': 'Financials', 'JPM': 'Financials', 'V': 'Financials',
    'LLY': 'Health Care', 'JNJ': 'Health Care',
    'AMZN': 'Consumer Discretionary', 'TSLA': 'Consumer Discretionary',
    'T': 'Communication Services', 'GOOGL': 'Communication Services',
    'GE': 'Industrials', 'CAT': 'Industrials',
    'WMT': 'Consumer Staples', 'COST': 'Consumer Staples',
    'XOM': 'Energy', 'NEE': 'Utilities', 'WELL': 'Real Estate', 'LIN': 'Materials',
}

correlations = returns[stocks].corr()      # each pair uses the days on which both stocks have data

rows = []
for stock in stocks + ['SPY']:
    r = returns[stock].dropna()
    wealth = (1 + r).cumprod()
    rows.append({
        'Ticker': stock,
        'Sector': sector.get(stock, 'ETF (benchmark)'),
        'First date': r.index.min().date(),
        'Ann. return (CAGR)': wealth.iloc[-1] ** (252 / len(r)) - 1,
        'Ann. volatility': r.std() * np.sqrt(252),
        'Max drawdown': (wealth / wealth.cummax() - 1).min(),
        'Avg. correlation with the other stocks': (correlations.loc[stock].sum() - 1) / (len(stocks) - 1) if stock in stocks else np.nan,
    })

universe_summary = pd.DataFrame(rows).set_index('Ticker')
display(universe_summary.round(3))

In [ ]:
# Heatmap of the correlations of daily returns
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(correlations.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(stocks)))
ax.set_xticklabels(stocks, rotation=90)
ax.set_yticks(range(len(stocks)))
ax.set_yticklabels(stocks)
for i in range(len(stocks)):
    for j in range(len(stocks)):
        ax.text(j, i, f'{correlations.values[i, j]:.1f}', ha='center', va='center', fontsize=7)
plt.colorbar(image, label='Correlation')
ax.set_title('Correlation of daily returns, 23 stocks (2010-2026)')
plt.tight_layout()
plt.savefig('../figures/04_universe_correlations.png', dpi=150)
plt.show()

print(f'Average correlation between pairs of stocks: {(correlations.values.sum() - len(stocks)) / (len(stocks) * (len(stocks) - 1)):.2f}')

### Risk-free rate: 3-year Treasury yield (FRED `DGS3`)

The code tries to read the file `data/risk_free_DGS3.csv`. If it does not exist yet, it downloads the series from FRED and saves it
(extraction date = the date on which this code is run; it is recorded in the file `data/risk_free_DGS3_info.txt`;
if the download fails and you download the CSV by hand, write down the download date).

- The series is in % per year (e.g. 4.25 = 4.25%) → we divide by 100.
- On days when the stock market is open but the bond market is closed (e.g. *Columbus Day*), we use the last known value.
- Daily rate = annual rate / 252.

In [ ]:
RF_FILE = 'data/risk_free_DGS3.csv'
FRED_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3'

if os.path.exists(RF_FILE):
    print('Reading the risk-free rate from the local file:', RF_FILE)
    rate = pd.read_csv(RF_FILE, index_col=0, parse_dates=True, na_values='.')   # FRED uses '.' on days with no value
else:
    print('Downloading the DGS3 series from FRED...')
    try:
        rate = pd.read_csv(FRED_URL, index_col=0, parse_dates=True, na_values='.')
    except Exception as error:
        # This happens, for example, on networks with a proxy/firewall (university, company)
        print('Could not download from FRED:', error)
        print('SOLUTION: open this link in your browser (it downloads a CSV):', FRED_URL)
        print('and save the file as:', os.path.abspath(RF_FILE))
        print('Then run this cell again.')
        print("Write down today's date: it is the extraction date of the risk-free rate (the report must state the source and the date).")
        raise
    rate.to_csv(RF_FILE)
    with open('data/risk_free_DGS3_info.txt', 'w') as f:
        f.write(f'Source: {FRED_URL}\nExtracted on: {pd.Timestamp.today().date()}\n')

# Keep only the column with the values, convert from % to decimal and drop days with no value
rate = rate.iloc[:, 0] / 100
rate = rate.dropna()

# Align with the stock trading days (method='ffill' = use the last known value)
rf_annual = rate.reindex(returns.index, method='ffill')
rf_daily = rf_annual / DAYS_PER_YEAR

# Check: the rate must exist on EVERY day (otherwise the Sharpe ratio and Max Sharpe would be wrong)
if rf_annual.isna().sum() > 0:
    raise ValueError('The risk-free rate does not cover the whole stock period (it starts on ' + str(rate.index.min().date()) +
                     '). Delete ' + RF_FILE + ' and download the COMPLETE series from FRED.')

# Check: the real DGS3 series moves over time (about 0.1% in 2020-2021 and above 4% in 2023).
# A constant series means the file is not the real data (e.g. a test file).
if rf_annual.min() == rf_annual.max():
    raise ValueError('The risk-free rate is constant, so ' + RF_FILE + ' is not the real DGS3 series. '
                     'Delete it and run this cell again to download it from FRED.')

print('Period of the rate:', rate.index.min().date(), 'to', rate.index.max().date())
print(f'Average rate over the period: {rf_annual.mean():.2%} | minimum: {rf_annual.min():.2%} | maximum: {rf_annual.max():.2%}')

plt.figure(figsize=(10, 3.5))
plt.plot(rf_annual.index, rf_annual * 100, color='darkgreen')
plt.title('Risk-free rate: 3-year US Treasury yield (FRED DGS3)')
plt.ylabel('% per year')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04_risk_free_rate.png', dpi=150)
plt.show()

## 3. The 8 strategies

Notation: $w$ = vector of weights, $\mu$ = annualised expected returns, $\Sigma$ = annualised covariance matrix,
$\sigma_i$ = volatility of asset *i*, $C$ = correlation matrix, $r_f$ = risk-free rate.
**Common constraints:** $w_i \ge 0$ and $\sum_i w_i = 1$.

| Strategy | Objective | Uses $\mu$? | Uses $\Sigma$? |
|---|---|---|---|
| EW | $w_i = 1/N$ | no | no |
| Markowitz (λ = 3) | $\max_w \; w^\top\mu - \frac{\lambda}{2}\, w^\top\Sigma w$ | yes | yes |
| GMV | $\min_w \; w^\top\Sigma w$ | no | yes |
| Max Sharpe | $\max_w \; (w^\top\mu - r_f)\,/\,\sqrt{w^\top\Sigma w}$ | yes | yes |
| Inverse vol | $w_i \propto 1/\sigma_i$ | no | only the diagonal |
| Risk parity (ERC) | equal risk contributions: $w_i(\Sigma w)_i / w^\top\Sigma w = 1/N$ | no | yes |
| Most diversified | $\max_w \; DR = w^\top\sigma \,/\, \sqrt{w^\top\Sigma w}$ | no | yes |
| Max decorrelation | $\min_w \; w^\top C\, w$ | no | only the correlations |

**Why is Markowitz different from GMV and Max Sharpe?** With λ = 3 the investor accepts some risk in exchange for return:
as λ → ∞ Markowitz gets closer to GMV; with a small λ it moves towards the asset with the highest expected return.
Max Sharpe is the point of the frontier that is tangent to the risk-free rate, which in general does not coincide with λ = 3 (see the chart in section 4).

**Inverse vol vs ERC:** both give more weight to the less volatile assets, but inverse vol **ignores correlations**.
They are only the same if all the correlations are equal. With different correlations, ERC gives less weight to the assets that are highly correlated with the others.

The optimisers use `scipy.optimize.minimize` with the **SLSQP** method (it accepts bounds and equality constraints).
If SLSQP reports a failure, the optimisation is repeated once with a looser tolerance (`ftol = 1e-8`); if it fails again,
equal weights are used at that rebalancing and the failure is recorded (`04_optimisation_failures.csv`, section 11).

In [ ]:
def estimate_parameters(sample):
    # ANNUALISED mean (vector) and covariance (matrix) from daily returns
    mu = sample.mean().values * DAYS_PER_YEAR
    sigma = sample.cov().values * DAYS_PER_YEAR
    return mu, sigma


failure_log = []   # here we store the optimisations that fail (the brief asks us not to hide them)
retry_log = []     # optimisations that failed at the first attempt but worked at the second one


def optimise(objective_function, n_assets, name, initial_weights=None, min_weight=0.0):
    # Minimises 'objective_function' subject to: min_weight <= w_i <= 1 and sum of weights = 1
    if initial_weights is None:
        initial_weights = np.ones(n_assets) / n_assets          # start from equally weighted

    bounds = [(min_weight, 1)] * n_assets
    sum_constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # ftol = precision requested from the optimiser. With values that are too small (e.g. 1e-12) SLSQP sometimes
    # is already at the optimum but reports a failure; 1e-10 is precise and almost always converges.
    result = minimize(objective_function, initial_weights, method='SLSQP',
                      bounds=bounds, constraints=[sum_constraint],
                      options={'maxiter': 1000, 'ftol': 1e-10})

    if not result.success:
        retry_log.append({'Strategy': name, 'Message': result.message})   # first attempt failed
        # Second attempt with a slightly looser precision (1e-8). Sometimes SLSQP stops right next to the optimum
        # and reports a failure; whether that happens can depend on the computer / library version, so this retry
        # makes the results more stable across computers.
        result = minimize(objective_function, initial_weights, method='SLSQP',
                          bounds=bounds, constraints=[sum_constraint],
                          options={'maxiter': 1000, 'ftol': 1e-8})

    if not result.success:
        # Plan B: if it still fails, we use equally weighted and record the failure
        failure_log.append({'Strategy': name, 'Message': result.message})
        return np.ones(n_assets) / n_assets

    weights = np.clip(result.x, 0, None)   # remove tiny numerical negatives (e.g. -1e-15)
    return weights / weights.sum()

Each strategy is a function that receives the **estimation sample** (daily returns of the rolling window) and the **annual risk-free rate**
and returns a vector of weights. Note: in Python, `@` is the matrix product (e.g. `w @ sigma @ w` = $w^\top\Sigma w$ = portfolio variance).

In [ ]:
# a) Equally weighted
def weights_equal_weight(sample, rf):
    n = sample.shape[1]
    return np.ones(n) / n


# b) Markowitz mean-variance with risk aversion lambda
def weights_markowitz(sample, rf, aversion=RISK_AVERSION):
    mu, sigma = estimate_parameters(sample)

    def objective(w):
        utility = w @ mu - aversion / 2 * (w @ sigma @ w)
        return -utility                     # minimising the negative = maximising the utility

    return optimise(objective, len(mu), 'Markowitz')


# c) Global minimum variance
def weights_min_variance(sample, rf):
    mu, sigma = estimate_parameters(sample)

    def objective(w):
        return w @ sigma @ w                # portfolio variance

    return optimise(objective, len(mu), 'GMV')


# d) Max Sharpe (tangency portfolio)
def weights_max_sharpe(sample, rf):
    mu, sigma = estimate_parameters(sample)

    def objective(w):
        sharpe = (w @ mu - rf) / np.sqrt(w @ sigma @ w)
        return -sharpe

    return optimise(objective, len(mu), 'Max Sharpe')

In [ ]:
# e1) Inverse volatility (no optimiser needed)
def weights_inverse_vol(sample, rf):
    vol = sample.std().values
    return (1 / vol) / np.sum(1 / vol)


def risk_contributions(w, sigma):
    # Share of the total portfolio risk that comes from each asset (sum = 1)
    variance = w @ sigma @ w
    return w * (sigma @ w) / variance


# e2) Risk parity / Equal Risk Contribution
def weights_risk_parity(sample, rf):
    mu, sigma = estimate_parameters(sample)
    n = len(mu)

    def objective(w):
        # sum of the squared differences between each asset's contribution and 1/N
        return np.sum((risk_contributions(w, sigma) - 1 / n) ** 2)

    # We start from the inverse vol weights (already close to the solution) and force every weight to be > 0,
    # because in ERC every asset must contribute to the risk.
    start_weights = weights_inverse_vol(sample, rf)
    return optimise(objective, n, 'Risk parity (ERC)', initial_weights=start_weights, min_weight=1e-6)

In [ ]:
def diversification_ratio(w, sigma):
    # DR = weighted average of the volatilities / portfolio volatility  (>= 1; higher = more diversified)
    vol = np.sqrt(np.diag(sigma))
    return (w @ vol) / np.sqrt(w @ sigma @ w)


# f1) Most diversified portfolio
def weights_most_diversified(sample, rf):
    mu, sigma = estimate_parameters(sample)

    def objective(w):
        return -diversification_ratio(w, sigma)

    return optimise(objective, len(mu), 'Most diversified')


# f2) Maximum decorrelation: it is the "minimum variance" but using the CORRELATION matrix
#     (as if every asset had the same volatility)
def weights_max_decorrelation(sample, rf):
    correlation = sample.corr().values

    def objective(w):
        return w @ correlation @ w

    return optimise(objective, correlation.shape[0], 'Max decorrelation')

In [ ]:
# Dictionary with all the strategies: name -> function
strategies = {
    'EW': weights_equal_weight,
    'Markowitz (λ=3)': weights_markowitz,
    'GMV': weights_min_variance,
    'Max Sharpe': weights_max_sharpe,
    'Inverse vol': weights_inverse_vol,
    'Risk parity (ERC)': weights_risk_parity,
    'Most diversified': weights_most_diversified,
    'Max decorrelation': weights_max_decorrelation,
}

# Order and colours used in every chart
order = list(strategies.keys()) + ['SPY (buy & hold)']
colours = dict(zip(order, plt.cm.tab10.colors))

## 4. Illustration: weights of each strategy with the last year of data (23 stocks)

Before the backtest, we look at what each strategy does with **all 23 stocks** and the last year of data
(as if we were rebalancing today). This helps to understand the differences between them.

In [ ]:
illustration_sample = returns[stocks].iloc[-ESTIMATION_WINDOW:]      # last year (252 days)
illustration_rf = rf_annual.iloc[-1]
print('Sample:', illustration_sample.index.min().date(), 'to', illustration_sample.index.max().date(),
      f'| risk-free rate = {illustration_rf:.2%}')

illustration_weights = pd.DataFrame(index=stocks)
for name, function in strategies.items():
    illustration_weights[name] = function(illustration_sample, illustration_rf)

display((100 * illustration_weights).round(1))

In [ ]:
# Ex-ante (in-sample) characteristics of each portfolio
mu_illus, sigma_illus = estimate_parameters(illustration_sample)

rows = []
for name in strategies.keys():
    w = illustration_weights[name].values
    exp_return = w @ mu_illus
    vol = np.sqrt(w @ sigma_illus @ w)
    rows.append({
        'Strategy': name,
        'Exp. return': exp_return,
        'Volatility': vol,
        'Sharpe (ex-ante)': (exp_return - illustration_rf) / vol,
        'Effective N (1/Σw²)': 1 / np.sum(w ** 2),
        'Max weight': w.max(),
        'N assets with w > 1%': int(np.sum(w > 0.01)),
        'Diversification ratio': diversification_ratio(w, sigma_illus),
    })

characteristics = pd.DataFrame(rows).set_index('Strategy')
display(characteristics.round(3))

In [ ]:
# Chart of the weights (one horizontal bar per strategy, stacked by stock)
fig, ax = plt.subplots(figsize=(13, 6))
left = np.zeros(len(strategies))
stock_colours = plt.cm.tab20(np.linspace(0, 1, len(stocks)))

for i, stock in enumerate(stocks):
    values = illustration_weights.loc[stock].values
    ax.barh(list(strategies.keys()), values, left=left, color=stock_colours[i], label=stock, edgecolor='white', linewidth=0.5)
    left = left + values

ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('Weight')
ax.set_title('Portfolio weights with the last year of data (23 stocks)')
ax.legend(ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../figures/04_weights_illustration.png', dpi=150)
plt.show()

### Inverse vol vs Risk parity (ERC): risk contributions

In inverse vol the **weights** are proportional to $1/\sigma_i$, but the **contributions to risk** are not equal,
because the assets that are more correlated with the rest of the portfolio contribute more. ERC corrects this.

In [ ]:
rc_iv = risk_contributions(illustration_weights['Inverse vol'].values, sigma_illus)
rc_erc = risk_contributions(illustration_weights['Risk parity (ERC)'].values, sigma_illus)

rc_table = pd.DataFrame({
    'Weight IV': illustration_weights['Inverse vol'],
    'Risk contrib. IV': rc_iv,
    'Weight ERC': illustration_weights['Risk parity (ERC)'],
    'Risk contrib. ERC': rc_erc,
    'Avg. correlation with others': (illustration_sample.corr().sum() - 1) / (len(stocks) - 1),
})
display((rc_table * 100).round(2).sort_values('Risk contrib. IV', ascending=False))
print(f'Standard deviation of the risk contributions -> IV: {rc_iv.std():.4f} | ERC: {rc_erc.std():.6f}')

x = np.arange(len(stocks))
plt.figure(figsize=(13, 4))
plt.bar(x - 0.2, rc_iv * 100, width=0.4, label='Inverse vol', color=colours['Inverse vol'])
plt.bar(x + 0.2, rc_erc * 100, width=0.4, label='Risk parity (ERC)', color=colours['Risk parity (ERC)'])
plt.axhline(100 / len(stocks), color='black', linestyle='--', linewidth=1, label='1/N')
plt.xticks(x, stocks, rotation=60)
plt.ylabel('% of portfolio risk')
plt.title('Risk contributions: inverse volatility vs equal risk contribution')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/04_risk_contributions_iv_vs_erc.png', dpi=150)
plt.show()

### Efficient frontier (in-sample)

We compute the long-only frontier: for each target return, the portfolio with the lowest variance. We mark the 8 strategies
and Markowitz with λ = 1, 3 and 5 to show that it is different from GMV and Max Sharpe.

In [ ]:
n = len(stocks)
gmv_return = illustration_weights['GMV'].values @ mu_illus
targets = np.linspace(gmv_return, mu_illus.max(), 40)

frontier_vol = []
for target in targets:
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                   {'type': 'eq', 'fun': lambda w: w @ mu_illus - target}]     # portfolio return = target
    res = minimize(lambda w: w @ sigma_illus @ w, np.ones(n) / n, method='SLSQP',
                   bounds=[(0, 1)] * n, constraints=constraints, options={'maxiter': 1000})
    frontier_vol.append(np.sqrt(res.fun))

plt.figure(figsize=(10, 6))
plt.plot(frontier_vol, targets, color='black', linewidth=1.5, label='Efficient frontier (long-only)')
plt.scatter(np.sqrt(np.diag(sigma_illus)), mu_illus, color='lightgrey', edgecolor='grey', s=30, label='Individual stocks')

for name in strategies.keys():
    w = illustration_weights[name].values
    plt.scatter(np.sqrt(w @ sigma_illus @ w), w @ mu_illus, color=colours[name], s=90, edgecolor='black', zorder=3, label=name)

for aversion in [1, 5]:
    w = weights_markowitz(illustration_sample, illustration_rf, aversion=aversion)
    plt.scatter(np.sqrt(w @ sigma_illus @ w), w @ mu_illus, color=colours['Markowitz (λ=3)'], marker='^', s=70, zorder=3)
    plt.annotate(f'λ={aversion}', (np.sqrt(w @ sigma_illus @ w), w @ mu_illus), textcoords='offset points', xytext=(6, -10))

plt.xlabel('Volatility (annualised)')
plt.ylabel('Expected return (annualised)')
plt.title('In-sample efficient frontier and strategy portfolios (last 12 months)')
plt.legend(fontsize=8, loc='best')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04_efficient_frontier.png', dpi=150)
plt.show()

**Warning:** this illustration is **in-sample** — it uses 1-year historical means as if they were the future.
The strategies that depend on $\mu$ (Markowitz and Max Sharpe) concentrate on the stocks that rose the most in the last year.
That is exactly why we need the out-of-sample backtest: to see whether this apparent advantage holds in the following year.

## 5. Design of the 100 experiments

For each experiment we draw:
1. **One contiguous 3-year period** (756 trading days) within 2010–2026 — the chronological order is kept;
2. **12 stocks** among those with complete data in that period (*eligibility*: TSLA is only eligible in periods that start after 29/06/2010).

We use the **same** sample (period + stocks) for every strategy, and the seed is recorded → the result is reproducible.

In [ ]:
# np.random.seed: with the same seed, NumPy guarantees the same sequence of numbers on any version/computer
np.random.seed(SEED)
dates = returns.index
n_days = len(dates)

rows = []
for e in range(1, N_EXPERIMENTS + 1):
    # 1) draw the first day of the 3-year period (the whole period must fit inside the data)
    start = int(np.random.randint(0, n_days - DATASET_DAYS + 1))
    end = start + DATASET_DAYS                        # exclusive

    # 2) eligible stocks = no missing values in the period
    block = returns[stocks].iloc[start:end]
    eligible = [stock for stock in stocks if block[stock].notna().all()]

    # 3) draw 12 stocks (without replacement)
    chosen = sorted(str(a) for a in np.random.choice(eligible, size=N_ASSETS, replace=False))

    rows.append({
        'Experiment': e,
        'start_index': start,
        'Estimation start': dates[start].date(),
        'Evaluation start': dates[start + EVAL_START].date(),
        'Evaluation end': dates[end - 1].date(),
        'N eligible': len(eligible),
        'Assets': chosen,
    })

design = pd.DataFrame(rows).set_index('Experiment')
display(design.head(10))
print('Experiments in which TSLA was not eligible:', (design['N eligible'] < len(stocks)).sum())

In [ ]:
# Chart 1: sampled periods (grey = estimation, blue = evaluation)
sorted_design = design.sort_values('start_index')

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2, 1]})
for y, (e, row) in enumerate(sorted_design.iterrows()):
    axes[0].plot([row['Estimation start'], row['Evaluation start']], [y, y], color='lightgrey', linewidth=2)
    axes[0].plot([row['Evaluation start'], row['Evaluation end']], [y, y], color='steelblue', linewidth=2)
axes[0].set_title('100 sampled 3-year periods (grey = estimation, blue = out-of-sample year)')
axes[0].set_ylabel('Experiment (sorted by start date)')

# Chart 2: how many times each stock was drawn
selection_count = pd.Series(0, index=stocks)
for asset_list in design['Assets']:
    for stock in asset_list:
        selection_count[stock] += 1
selection_count.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Times each stock was selected')
axes[1].set_xlabel('Number of experiments')

plt.tight_layout()
plt.savefig('../figures/04_experiment_design.png', dpi=150)
plt.show()

**Important note:** the sampled periods **overlap** (there are only ~16 years of data, so there are at most ~14 different evaluation years).
The 100 experiments **are not 100 independent tests** — they are used to look at the **distribution** of the results and at whether the conclusions
depend on the period or on the stocks drawn, and not to run significance tests as if they were independent.

## 6. Simulating one strategy in one experiment

For each experiment and strategy:

1. **Rebalancing dates:** every 126 days. The first one is **one semester before** the start of the evaluation (warm-up):
   this way, on the 1st evaluation day the portfolio already exists and the turnover on that day is realistic (it is not a purchase made from cash).
   The warm-up returns are **not** included in the metrics.
2. At each rebalancing, the weights are computed **using only the previous 252 days** (no look-ahead).
   Trades are made at the close of the previous day, i.e. the new weights apply from day *t* onwards
   (*market-on-close* convention: we trade at the closing price of the day on which we know the last observation).
3. **Turnover** = $\sum_i |w_i^{target} - w_i^{before}|$ (purchases + sales, as a % of the portfolio).
4. Between rebalancings the weights **drift**: $w_{i,t+1} = w_{i,t}(1+R_{i,t}) / (1+R_{p,t})$.
5. **Net return** on the rebalancing day = gross return − cost × turnover.

In [ ]:
def simulate(block_returns, rf_block, weight_function, interval=REBALANCE_INTERVAL, window=ESTIMATION_WINDOW):
    # block_returns: DataFrame with 756 days x 12 stocks (simple returns)
    # rf_block: Series with 756 days of ANNUAL risk-free rate
    # Returns a dictionary with the gross returns and the turnover of each day of the evaluation year,
    # and the target weights / diversification ratio of each rebalancing in the evaluation year.
    R = block_returns.values                 # numpy is faster inside a loop
    n_assets = R.shape[1]

    # Rebalancing days (the 1st is one interval before the start of the evaluation = warm-up)
    rebalance_days = list(range(EVAL_START - interval, DATASET_DAYS, interval))
    if rebalance_days[0] - window < 0:
        raise ValueError('The estimation window does not fit before the 1st rebalancing')

    weights = np.zeros(n_assets)            # we start in cash
    gross_returns = []
    daily_turnover = []
    target_weights = []
    div_ratios = []

    for t in range(rebalance_days[0], DATASET_DAYS):
        turnover = 0.0

        if t in rebalance_days:
            # 1) estimate using only PAST data (days t-window to t-1)
            sample = block_returns.iloc[t - window:t]
            known_rf = rf_block.iloc[t - 1]
            n_failures_before = len(failure_log)
            target = weight_function(sample, known_rf)
            for failure in failure_log[n_failures_before:]:          # if the optimisation failed, store the date
                failure['Rebalance date'] = block_returns.index[t].date()

            # 2) how much we need to trade to move from the current weights to the target
            turnover = np.sum(np.abs(target - weights))
            weights = target

            if t >= EVAL_START:
                target_weights.append(target)
                div_ratios.append(diversification_ratio(target, sample.cov().values))

        # 3) portfolio return on day t
        r_portfolio = np.sum(weights * R[t])

        # 4) the weights drift with prices until the next rebalancing
        weights = weights * (1 + R[t]) / (1 + r_portfolio)

        # 5) we only store the evaluation year
        if t >= EVAL_START:
            gross_returns.append(r_portfolio)
            daily_turnover.append(turnover)

    eval_dates = block_returns.index[EVAL_START:]
    return {
        'r_gross': pd.Series(gross_returns, index=eval_dates),
        'turnover': pd.Series(daily_turnover, index=eval_dates),
        'weights': target_weights,
        'div_ratio': div_ratios,
    }

## 7. Performance metrics (conventions)

Computed on the **daily** returns of the evaluation year (252 days):

| Metric | Formula / convention |
|---|---|
| Annualised return | $\left(\prod_t (1+R_t)\right)^{252/n} - 1$ (geometric) |
| Annualised volatility | $\text{standard deviation}(R_t)\times\sqrt{252}$ |
| Sharpe | $\overline{(R_t - r_{f,t})} \,/\, \text{sd}(R_t - r_{f,t}) \times \sqrt{252}$ |
| Sortino | $\overline{(R_t - r_{f,t})} \,/\, DD \times \sqrt{252}$, with $DD = \sqrt{\text{mean}(\min(R_t - r_{f,t}, 0)^2)}$ |
| Maximum drawdown (MDD) | largest fall from a previous peak of wealth (which starts at 1) |
| Calmar | annualised return / \|MDD\| |
| Sterling | annualised return / (\|MDD\| + 10%) — original version; with 1 evaluation year the "average annual MDD" is the MDD of that year |
| Annual turnover | sum of the turnover of the rebalancings in the evaluation year |
| Cost drag | gross annualised return − net annualised return |
| Effective N | $1/\sum_i w_i^2$ (average across rebalancings): 12 = equal weights, 1 = everything in one stock |
| Max weight | largest target weight in the portfolio (average across rebalancings) |
| Diversification ratio (ex-ante) | $w^\top\sigma / \sqrt{w^\top\Sigma w}$ computed with the **estimation sample** (the 252 days before the rebalancing), averaged across rebalancings. It is an *ex-ante* (in-sample) measure, not an out-of-sample result — *Most diversified* has the highest value by construction |

The ratios use the **risk-free rate of each day** (Sharpe, Sortino). Calmar and Sterling use the annualised return (without subtracting the risk-free rate), as in the original definitions.

In [ ]:
def compute_metrics(r, rf_d):
    # r: daily portfolio returns | rf_d: daily risk-free rate on the same dates
    excess = r - rf_d
    n = len(r)

    wealth = (1 + r).cumprod()
    ann_return = wealth.iloc[-1] ** (DAYS_PER_YEAR / n) - 1
    ann_vol = r.std() * np.sqrt(DAYS_PER_YEAR)

    sharpe = excess.mean() / excess.std() * np.sqrt(DAYS_PER_YEAR)

    downside_dev = np.sqrt(np.mean(np.minimum(excess, 0) ** 2))
    sortino = excess.mean() / downside_dev * np.sqrt(DAYS_PER_YEAR)

    # Drawdown: we include the initial value 1 so that a fall on the very 1st day is counted
    wealth_with_start = np.concatenate([[1.0], wealth.values])
    running_max = np.maximum.accumulate(wealth_with_start)
    drawdown = wealth_with_start / running_max - 1
    mdd = drawdown.min()                                    # negative number

    calmar = ann_return / abs(mdd) if mdd < 0 else np.nan
    sterling = ann_return / (abs(mdd) + 0.10)

    return {
        'Ann. return': ann_return,
        'Ann. volatility': ann_vol,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max drawdown': mdd,
        'Calmar': calmar,
        'Sterling': sterling,
    }

In [ ]:
def metrics_table(simulations, cost, design_table=None):
    # Turns the simulations into a table: one row per (experiment, strategy)
    # design_table: by default uses the main design of the experiments
    if design_table is None:
        design_table = design

    rows = []
    for (e, name), sim in simulations.items():
        rf_d = rf_daily.loc[sim['r_gross'].index]

        r_net = sim['r_gross'] - cost * sim['turnover']
        m_net = compute_metrics(r_net, rf_d)
        m_gross = compute_metrics(sim['r_gross'], rf_d)

        row = {'Experiment': e, 'Strategy': name}
        row.update(m_net)                                  # metrics net of costs
        row['Ann. return (gross)'] = m_gross['Ann. return']
        row['Sharpe (gross)'] = m_gross['Sharpe']
        row['Cost drag'] = m_gross['Ann. return'] - m_net['Ann. return']
        row['Turnover (annual)'] = sim['turnover'].sum()
        row['Effective N'] = np.mean([1 / np.sum(w ** 2) for w in sim['weights']])
        row['Max weight'] = np.mean([w.max() for w in sim['weights']])
        row['Diversification ratio (ex-ante)'] = np.mean(sim['div_ratio'])
        rows.append(row)

    # Market benchmark: SPY bought at the start of the evaluation year and held (no rebalancing costs)
    for e in design_table.index:
        start = design_table.loc[e, 'start_index']
        eval_dates = returns.index[start + EVAL_START: start + DATASET_DAYS]
        m_spy = compute_metrics(returns.loc[eval_dates, 'SPY'], rf_daily.loc[eval_dates])
        row = {'Experiment': e, 'Strategy': 'SPY (buy & hold)'}
        row.update(m_spy)
        row['Ann. return (gross)'] = m_spy['Ann. return']
        row['Sharpe (gross)'] = m_spy['Sharpe']
        row['Cost drag'] = 0.0
        row['Turnover (annual)'] = 0.0
        rows.append(row)

    return pd.DataFrame(rows)


def run_backtest(strategy_dict, interval=REBALANCE_INTERVAL, design_table=None, run_name='Main'):
    # Runs every strategy in every experiment. Returns a dictionary (experiment, strategy) -> simulation
    # run_name: identifies this run in the optimisation failure log
    if design_table is None:
        design_table = design

    simulations = {}
    for e in design_table.index:
        start = design_table.loc[e, 'start_index']
        assets_e = design_table.loc[e, 'Assets']
        block = returns[assets_e].iloc[start:start + DATASET_DAYS]
        rf_block = rf_annual.iloc[start:start + DATASET_DAYS]

        for name, function in strategy_dict.items():
            n_failures_before = len(failure_log)
            simulations[(e, name)] = simulate(block, rf_block, function, interval=interval)
            for failure in failure_log[n_failures_before:]:       # identify the failures of this simulation
                failure['Run'] = run_name
                failure['Experiment'] = e
                failure['Strategy'] = name                        # full name, e.g. 'Markowitz (λ=1)'
    return simulations


def show_failures(run_name):
    # Shows the optimisations that failed in one run (on those rebalancings we used equal weights)
    failures = [f for f in failure_log if f.get('Run') == run_name]
    print(f'Optimisations that failed ({run_name}): {len(failures)}')
    if len(failures) > 0:
        display(pd.DataFrame(failures))

## 8. Running the main backtest (semi-annual rebalancing, 10 bps)

In [ ]:
import time
clock_start = time.time()

simulations = run_backtest(strategies, run_name='Main (semi-annual)')
results = metrics_table(simulations, COST)

print(f'Backtest finished in {time.time() - clock_start:.0f} seconds')
print('Number of simulations:', len(simulations), '| rows in the table (including SPY):', len(results))
show_failures('Main (semi-annual)')

In [ ]:
display(results.head(10).round(4))

## 9. Results across the 100 experiments

### 9.1 Median of each metric (net of costs, 10 bps)

In [ ]:
summary_columns = ['Ann. return', 'Ann. volatility', 'Sharpe', 'Sortino', 'Max drawdown', 'Calmar', 'Sterling',
                   'Ann. return (gross)', 'Sharpe (gross)', 'Cost drag', 'Turnover (annual)',
                   'Effective N', 'Max weight', 'Diversification ratio (ex-ante)']

medians = results.groupby('Strategy')[summary_columns].median().loc[order]
display(medians.round(3))

### 9.2 Dispersion across experiments and % of experiments in which each strategy beats EW

In [ ]:
# Dispersion: 25th and 75th percentiles of the Sharpe ratio, the return and the Sterling ratio (net)
dispersion = pd.DataFrame({
    'Sharpe P25': results.groupby('Strategy')['Sharpe'].quantile(0.25),
    'Sharpe median': results.groupby('Strategy')['Sharpe'].median(),
    'Sharpe P75': results.groupby('Strategy')['Sharpe'].quantile(0.75),
    'Return P25': results.groupby('Strategy')['Ann. return'].quantile(0.25),
    'Return median': results.groupby('Strategy')['Ann. return'].median(),
    'Return P75': results.groupby('Strategy')['Ann. return'].quantile(0.75),
    'Return std': results.groupby('Strategy')['Ann. return'].std(),
    'Sterling P25': results.groupby('Strategy')['Sterling'].quantile(0.25),
    'Sterling median': results.groupby('Strategy')['Sterling'].median(),
    'Sterling P75': results.groupby('Strategy')['Sterling'].quantile(0.75),
}).loc[order]
display(dispersion.round(3))


def pct_beats_ew(table):
    # % of experiments in which the strategy is better than EW in the SAME experiment
    result = pd.DataFrame(index=order)
    for metric, higher_is_better in [('Sharpe', True), ('Ann. return', True), ('Max drawdown', True),
                                     ('Ann. volatility', False), ('Sterling', True)]:
        wide = table.pivot(index='Experiment', columns='Strategy', values=metric)   # experiments x strategies
        if higher_is_better:
            beats = wide.gt(wide['EW'], axis=0)       # gt = greater than
        else:
            beats = wide.lt(wide['EW'], axis=0)       # lt = lower than (less volatility is better)
        result[f'% beats EW: {metric}'] = 100 * beats.mean()
    return result.drop(index='EW')


beats_ew = pct_beats_ew(results)
display(beats_ew.round(0))

In [ ]:
# Boxplots of the main metrics across the 100 experiments
plot_metrics = [('Sharpe', 'Sharpe ratio (net)'), ('Ann. return', 'Annualised return (net)'),
                ('Ann. volatility', 'Annualised volatility'), ('Max drawdown', 'Maximum drawdown'),
                ('Sortino', 'Sortino ratio (net)'), ('Sterling', 'Sterling ratio (net)')]

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
for ax, (metric, title) in zip(axes.flatten(), plot_metrics):
    data = [results.loc[results['Strategy'] == name, metric].values for name in order]
    box = ax.boxplot(data, patch_artist=True, showfliers=True, medianprops=dict(color='black'))
    for patch, name in zip(box['boxes'], order):
        patch.set_facecolor(colours[name])
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order, rotation=35, ha='right')
    ax.axhline(np.median(data[0]), color='grey', linestyle='--', linewidth=1)   # EW median
    ax.set_title(f'{title} - 100 experiments (dashed = EW median)')
    ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/04_boxplots_metrics.png', dpi=150)
plt.show()

In [ ]:
# Return vs risk (medians) and % of experiments that beat EW
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name in order:
    axes[0].scatter(medians.loc[name, 'Ann. volatility'], medians.loc[name, 'Ann. return'],
                    s=120, color=colours[name], edgecolor='black', zorder=3)
    axes[0].annotate(name, (medians.loc[name, 'Ann. volatility'], medians.loc[name, 'Ann. return']),
                     textcoords='offset points', xytext=(7, 4), fontsize=9)
axes[0].set_xlabel('Median annualised volatility')
axes[0].set_ylabel('Median annualised return (net)')
axes[0].set_title('Median risk and return across 100 experiments')
axes[0].grid(True, linestyle='--', alpha=0.3)

beats_ew[['% beats EW: Sharpe', '% beats EW: Ann. return', '% beats EW: Max drawdown', '% beats EW: Sterling']].plot(
    kind='bar', ax=axes[1], color=['steelblue', 'darkorange', 'seagreen', 'purple'])
axes[1].axhline(50, color='black', linestyle='--', linewidth=1)
axes[1].set_ylabel('% of experiments')
axes[1].set_title('Share of experiments in which each strategy beats EW (net of costs)')
axes[1].set_xticklabels(beats_ew.index, rotation=35, ha='right')
axes[1].legend(['Sharpe', 'Return', 'Drawdown (smaller)', 'Sterling'], fontsize=9)

plt.tight_layout()
plt.savefig('../figures/04_risk_return_and_beats_ew.png', dpi=150)
plt.show()

### 9.3 Turnover, costs and concentration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
strategies_no_spy = list(strategies.keys())

medians.loc[strategies_no_spy, 'Turnover (annual)'].plot(kind='bar', ax=axes[0], color=[colours[n] for n in strategies_no_spy])
axes[0].set_title('Median annual turnover (Σ|Δw|)')

(10000 * medians.loc[strategies_no_spy, 'Cost drag']).plot(kind='bar', ax=axes[1], color=[colours[n] for n in strategies_no_spy])
axes[1].set_title('Median cost drag (bps per year, 10 bps cost)')

medians.loc[strategies_no_spy, 'Effective N'].plot(kind='bar', ax=axes[2], color=[colours[n] for n in strategies_no_spy])
axes[2].axhline(N_ASSETS, color='black', linestyle='--', linewidth=1)
axes[2].set_title(f'Median effective number of stocks (max = {N_ASSETS})')

for ax in axes:
    ax.set_xticklabels(strategies_no_spy, rotation=35, ha='right')
    ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/04_turnover_costs_concentration.png', dpi=150)
plt.show()

### 9.4 Ranking of the strategies in each experiment

In each experiment we rank the strategies by net Sharpe ratio (1 = best). A low and stable average rank is a good sign.

In [ ]:
sharpe_wide = results.pivot(index='Experiment', columns='Strategy', values='Sharpe')[order]
rankings = sharpe_wide.rank(axis=1, ascending=False)          # rank within each row (experiment)

rankings_table = pd.DataFrame({
    'Average rank': rankings.mean(),
    'Median rank': rankings.median(),
    '% ranked 1st': 100 * (rankings == 1).mean(),
    '% in top 3': 100 * (rankings <= 3).mean(),
    '% ranked last': 100 * (rankings == len(order)).mean(),
}).sort_values('Average rank')
display(rankings_table.round(1))

### 9.5 Example: evolution of wealth in one experiment

We choose the first experiment whose evaluation year includes the March 2020 *crash* (if there is one), to see the behaviour during a crisis.

In [ ]:
crisis_date = pd.Timestamp('2020-03-16')
candidates = [e for e in design.index
              if pd.Timestamp(design.loc[e, 'Evaluation start']) <= crisis_date <= pd.Timestamp(design.loc[e, 'Evaluation end'])]
example_exp = candidates[0] if len(candidates) > 0 else design.index[0]
print('Chosen experiment:', example_exp)
print('Stocks:', design.loc[example_exp, 'Assets'])
print('Evaluation:', design.loc[example_exp, 'Evaluation start'], 'to', design.loc[example_exp, 'Evaluation end'])

plt.figure(figsize=(12, 5))
for name in strategies.keys():
    sim = simulations[(example_exp, name)]
    r_net = sim['r_gross'] - COST * sim['turnover']
    plt.plot((1 + r_net).cumprod(), label=name, color=colours[name], linewidth=1.3)

example_dates = simulations[(example_exp, 'EW')]['r_gross'].index
plt.plot((1 + returns.loc[example_dates, 'SPY']).cumprod(), label='SPY (buy & hold)', color='black', linestyle='--', linewidth=1.3)
plt.axhline(1, color='grey', linewidth=0.8)
plt.title(f'Experiment {example_exp}: growth of 1 USD in the out-of-sample year (net of costs)')
plt.ylabel('Wealth')
plt.legend(fontsize=8, ncol=3)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04_wealth_example.png', dpi=150)
plt.show()

## 10. Robustness

### 10.1 Higher transaction costs (0, 10, 25 and 50 bps)

There is no need to simulate again: we just recompute the net returns with a different cost.

In [ ]:
cost_levels = [0.0, 0.0010, 0.0025, 0.0050]

sharpe_by_cost = pd.DataFrame(index=order)
ranking_by_cost = pd.DataFrame(index=order)
for cost in cost_levels:
    table_c = metrics_table(simulations, cost)
    label = f'{cost * 10000:.0f} bps'
    sharpe_by_cost[label] = table_c.groupby('Strategy')['Sharpe'].median()
    ranking_by_cost[label] = sharpe_by_cost[label].rank(ascending=False)

print('Median Sharpe ratio (net) for each cost level')
display(sharpe_by_cost.round(3))
print('Position in the ranking (by median Sharpe ratio)')
display(ranking_by_cost.astype(int))

plt.figure(figsize=(10, 5))
for name in order:
    plt.plot(cost_levels, sharpe_by_cost.loc[name], marker='o', color=colours[name], label=name)
plt.xticks(cost_levels, [f'{c * 10000:.0f}' for c in cost_levels])
plt.xlabel('Transaction cost (bps per unit traded)')
plt.ylabel('Median Sharpe ratio (net)')
plt.title('Sensitivity of the median Sharpe ratio to transaction costs')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/04_cost_sensitivity.png', dpi=150)
plt.show()

### 10.2 Quarterly rebalancing (convention suggested in the brief)

Same experiments, same strategies, but rebalancing every 63 days.

In [ ]:
simulations_quarterly = run_backtest(strategies, interval=63, run_name='Quarterly')
results_quarterly = metrics_table(simulations_quarterly, COST)
show_failures('Quarterly')

rebalance_comparison = pd.DataFrame({
    'Sharpe (semi-annual)': results.groupby('Strategy')['Sharpe'].median(),
    'Sharpe (quarterly)': results_quarterly.groupby('Strategy')['Sharpe'].median(),
    'Return (semi-annual)': results.groupby('Strategy')['Ann. return'].median(),
    'Return (quarterly)': results_quarterly.groupby('Strategy')['Ann. return'].median(),
    'Turnover (semi-annual)': results.groupby('Strategy')['Turnover (annual)'].median(),
    'Turnover (quarterly)': results_quarterly.groupby('Strategy')['Turnover (annual)'].median(),
}).loc[order]
rebalance_comparison['% beats EW Sharpe (quarterly)'] = pct_beats_ew(results_quarterly)['% beats EW: Sharpe']
display(rebalance_comparison.round(3))

### 10.3 Markowitz risk aversion (λ = 1, 3, 5)

In [ ]:
lambda_strategies = {
    'Markowitz (λ=1)': lambda sample, rf: weights_markowitz(sample, rf, aversion=1),
    'Markowitz (λ=5)': lambda sample, rf: weights_markowitz(sample, rf, aversion=5),
}
lambda_simulations = run_backtest(lambda_strategies, run_name='Markowitz lambda 1 and 5')
show_failures('Markowitz lambda 1 and 5')

# We add them to the simulations already done (λ=3, GMV and Max Sharpe) to compare
simulations_to_compare = {}
for (e, name), sim in simulations.items():
    if name in ['Markowitz (λ=3)', 'GMV', 'Max Sharpe', 'EW']:
        simulations_to_compare[(e, name)] = sim
simulations_to_compare.update(lambda_simulations)

lambda_table = metrics_table(simulations_to_compare, COST)
lambda_table = lambda_table[lambda_table['Strategy'] != 'SPY (buy & hold)']
lambda_order = ['EW', 'GMV', 'Markowitz (λ=5)', 'Markowitz (λ=3)', 'Markowitz (λ=1)', 'Max Sharpe']
lambda_sensitivity = lambda_table.groupby('Strategy')[['Ann. return', 'Ann. volatility', 'Sharpe', 'Max drawdown',
                                                       'Turnover (annual)', 'Effective N']].median().loc[lambda_order]
display(lambda_sensitivity.round(3))

### 10.4 Do the results depend on the period?

(a) Sharpe ratio difference relative to EW in each experiment, over time (start date of the evaluation).
(b) Medians by sub-period and by market regime (evaluation year with SPY going up or down).

In [ ]:
diff_vs_ew = sharpe_wide.sub(sharpe_wide['EW'], axis=0)        # strategy Sharpe - EW Sharpe
start_dates = pd.to_datetime(design['Evaluation start'])

plot_strategies = ['Markowitz (λ=3)', 'GMV', 'Max Sharpe', 'Risk parity (ERC)', 'Most diversified', 'Max decorrelation']
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharey=True)
for ax, name in zip(axes.flatten(), plot_strategies):
    ax.scatter(start_dates, diff_vs_ew[name], color=colours[name], s=20)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{name} minus EW (Sharpe)')
    ax.grid(True, linestyle='--', alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel('Start of out-of-sample year')
plt.tight_layout()
plt.savefig('../figures/04_sharpe_diff_vs_ew_over_time.png', dpi=150)
plt.show()

In [ ]:
def sub_period(date):
    # Groups the start of the evaluation year into 3 sub-periods
    if date.year < 2017:
        return '2012-2016'
    elif date.year < 2021:
        return '2017-2020'
    else:
        return '2021-2026'


# Market regime: did SPY go up or down in the evaluation year?
spy_return = results[results['Strategy'] == 'SPY (buy & hold)'].set_index('Experiment')['Ann. return']
regime = pd.Series(np.where(spy_return > 0, 'SPY up', 'SPY down'), index=spy_return.index)
period = start_dates.apply(sub_period)

results['Sub-period'] = results['Experiment'].map(period)
results['Market regime'] = results['Experiment'].map(regime)

print('Number of experiments per sub-period:', period.value_counts().sort_index().to_dict())
print('Number of experiments per regime:', regime.value_counts().to_dict())

sharpe_period = pd.pivot_table(results, index='Strategy', columns='Sub-period', values='Sharpe', aggfunc='median').loc[order]
sharpe_regime = pd.pivot_table(results, index='Strategy', columns='Market regime', values='Sharpe', aggfunc='median').loc[order]
return_regime = pd.pivot_table(results, index='Strategy', columns='Market regime', values='Ann. return', aggfunc='median').loc[order]

print('\nMedian Sharpe ratio by sub-period')
display(sharpe_period.round(3))
print('Median Sharpe ratio by market regime')
display(sharpe_regime.round(3))
print('Median return by market regime')
display(return_regime.round(3))

### 10.5 Do the results depend on the stocks drawn?

Average weight given to each stock by each strategy (average across rebalancings, **only in the experiments in which the stock was drawn**).
It shows where the money goes and whether the strategies depend on a few stocks.

In [ ]:
weight_sum = pd.DataFrame(0.0, index=stocks, columns=list(strategies.keys()))
weight_count = pd.DataFrame(0, index=stocks, columns=list(strategies.keys()))

for (e, name), sim in simulations.items():
    assets_e = design.loc[e, 'Assets']
    for w in sim['weights']:                    # one weight vector per rebalancing
        for stock, weight in zip(assets_e, w):
            weight_sum.loc[stock, name] += weight
            weight_count.loc[stock, name] += 1

avg_weight = weight_sum / weight_count

fig, ax = plt.subplots(figsize=(11, 10))
image = ax.imshow(avg_weight.values * 100, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(avg_weight.columns)))
ax.set_xticklabels(avg_weight.columns, rotation=35, ha='right')
ax.set_yticks(range(len(stocks)))
ax.set_yticklabels(stocks)
for i in range(len(stocks)):
    for j in range(len(avg_weight.columns)):
        value = avg_weight.values[i, j] * 100
        ax.text(j, i, f'{value:.0f}', ha='center', va='center', fontsize=8, color='white' if value > 20 else 'black')
plt.colorbar(image, label='Average weight when selected (%)')
ax.set_title('Average weight of each stock when selected (%) - equal weight would be 8.3%')
plt.tight_layout()
plt.savefig('../figures/04_avg_weight_by_stock.png', dpi=150)
plt.show()

### 10.6 Same periods, different stocks

To see whether the conclusions depend on the stocks drawn, we repeat the backtest with the **same 100 periods** but:
- (a) a **new draw of 12 stocks** (a different seed). As there are only 23 stocks, the two draws have on average about half of their stocks in common;
- (b) **all the eligible stocks** (22 or 23), i.e. without drawing stocks.

If the ranking of the strategies changes a lot between the three versions, the conclusions depend on the subset of stocks.

In [ ]:
# (a) Same periods, DIFFERENT 12 stocks (a different seed, used only to draw the stocks)
np.random.seed(SEED + 1)
rows = []
for e in design.index:
    start = design.loc[e, 'start_index']
    block = returns[stocks].iloc[start:start + DATASET_DAYS]
    eligible = [stock for stock in stocks if block[stock].notna().all()]
    chosen = sorted(str(a) for a in np.random.choice(eligible, size=N_ASSETS, replace=False))
    rows.append({'Experiment': e, 'start_index': start, 'Assets': chosen})
design_new_draw = pd.DataFrame(rows).set_index('Experiment')

in_common = [len(set(a) & set(b)) for a, b in zip(design['Assets'], design_new_draw['Assets'])]
print('Average number of stocks in common with the main draw:', round(np.mean(in_common), 1), 'out of', N_ASSETS)

# (b) Same periods, ALL the eligible stocks
rows = []
for e in design.index:
    start = design.loc[e, 'start_index']
    block = returns[stocks].iloc[start:start + DATASET_DAYS]
    eligible = [stock for stock in stocks if block[stock].notna().all()]
    rows.append({'Experiment': e, 'start_index': start, 'Assets': eligible})
design_all = pd.DataFrame(rows).set_index('Experiment')

simulations_new_draw = run_backtest(strategies, design_table=design_new_draw, run_name='New draw of 12 stocks')
results_new_draw = metrics_table(simulations_new_draw, COST, design_table=design_new_draw)
show_failures('New draw of 12 stocks')

simulations_all = run_backtest(strategies, design_table=design_all, run_name='All eligible stocks')
results_all = metrics_table(simulations_all, COST, design_table=design_all)
show_failures('All eligible stocks')

In [ ]:
def short_summary(table):
    # Median Sharpe ratio, position in the ranking, median return and % of experiments that beat EW
    median_sharpe = table.groupby('Strategy')['Sharpe'].median()
    return pd.DataFrame({
        'Median Sharpe': median_sharpe,
        'Rank': median_sharpe.rank(ascending=False),
        'Median return': table.groupby('Strategy')['Ann. return'].median(),
        '% beats EW (Sharpe)': pct_beats_ew(table)['% beats EW: Sharpe'],
    })


subset_comparison = pd.concat({
    '12 stocks (main)': short_summary(results),
    '12 stocks (new draw)': short_summary(results_new_draw),
    'All eligible stocks': short_summary(results_all),
}, axis=1).loc[order]
display(subset_comparison.round(3))

# Agreement: in each experiment, does the strategy beat (or not) EW in BOTH samples of 12 stocks?
sharpe_main = results.pivot(index='Experiment', columns='Strategy', values='Sharpe')
sharpe_new_draw = results_new_draw.pivot(index='Experiment', columns='Strategy', values='Sharpe')
beats_main = sharpe_main.gt(sharpe_main['EW'], axis=0)
beats_new_draw = sharpe_new_draw.gt(sharpe_new_draw['EW'], axis=0)
agreement = 100 * (beats_main == beats_new_draw).mean()
print('% of experiments with the same conclusion "beats / does not beat EW" in the two samples of stocks:')
display(agreement.drop(index=['EW', 'SPY (buy & hold)']).round(0).to_frame('Agreement (%)'))

## 11. Saving the results (CSV + Excel)

- `04_experiment_design.csv` — periods and stocks of each experiment (makes it possible to reproduce and reuse them in the individual project);
- `04_metrics_by_experiment.csv` — all the metrics, by experiment and strategy;
- `04_optimisation_failures.csv` — every optimisation that failed twice (with `ftol` 1e-10 and then 1e-8); on those rebalancings we used equal weights; it may be empty;
- `04_strategies_backtest.xlsx` — all the summary tables, one per sheet (**use this file to copy tables into the report**).

In [ ]:
# Note (Windows): close the .xlsx/.csv files in Excel before running this cell,
# otherwise Excel locks them and you get a "PermissionError".

design_csv = design.copy()
design_csv['Assets'] = [', '.join(asset_list) for asset_list in design_csv['Assets']]
results.to_csv('../results/04_metrics_by_experiment.csv', index=False, encoding='utf-8-sig')

# Failure log: the failures without a 'Run' came from the illustration in section 4
for failure in failure_log:
    if 'Run' not in failure:
        failure['Run'] = 'Illustration (section 4)'
failures_table = pd.DataFrame(failure_log, columns=['Run', 'Experiment', 'Rebalance date', 'Strategy', 'Message'])
print('Total number of optimisations that failed (all runs):', len(failures_table))

tables = {
    'universe_summary': universe_summary,
    'weights_illustration': illustration_weights,
    'illustration_characteristics': characteristics,
    'iv_vs_erc': rc_table,
    'median_metrics': medians,
    'dispersion': dispersion,
    'beats_ew': beats_ew,
    'rankings': rankings_table,
    'sharpe_by_cost': sharpe_by_cost,
    'ranking_by_cost': ranking_by_cost,
    'rebalance_semiann_vs_quarterly': rebalance_comparison,
    'lambda_sensitivity': lambda_sensitivity,
    'sharpe_by_subperiod': sharpe_period,
    'sharpe_by_regime': sharpe_regime,
    'return_by_regime': return_regime,
    'avg_weight_by_stock': avg_weight,
    'subset_comparison': subset_comparison,
    'experiment_design': design_csv,
    'optimisation_failures': failures_table,
}

with pd.ExcelWriter('../results/04_strategies_backtest.xlsx') as excel:
    for name, table in tables.items():
        table.to_excel(excel, sheet_name=name[:31])          # Excel accepts at most 31 characters in a sheet name
        table.to_csv(f'../results/04_{name}.csv', encoding='utf-8-sig')   # 'utf-8-sig' -> Excel displays λ and σ correctly

print('Results saved in ../results/')

## 12. Limitations (for the robustness section of the report)

- **Survivorship and selection bias:** the 23 stocks were chosen based on their **current** market capitalisation (September 2026).
  They are companies that survived and grew (NVDA, AVGO, LLY, …), which **inflates the absolute returns of every strategy**
  (EW included) compared with what an investor would have obtained in real time. The comparisons **between** strategies are less affected,
  but strategies that concentrate on past winners (Max Sharpe, Markowitz) may benefit from this bias.
- **Overlapping experiments:** the 100 experiments share evaluation years → they are not independent; we look at medians,
  dispersion and consistency and not at significance tests.
- **Estimation:** 1-year historical means are very noisy (estimation error) — this mainly affects Markowitz and Max Sharpe.
- **Simplified costs:** fixed proportional cost, with no market impact and no taxes; SPY also has a management fee (~0.09%/year) that is already included in its price.
- **Risk-free rate:** the 3-year Treasury carries some interest-rate risk; a 3-month T-bill would be the "purer" alternative.
- **Universe:** only large-cap US stocks — the results may not apply to other asset classes.

## 13. Automatic summary of the results

This cell writes the main conclusions **with the numbers from this run** (so that the text of the report matches the tables).

In [ ]:
strategies_no_spy = list(strategies.keys())
med = medians.loc[strategies_no_spy]          # medians from section 9.1, only the 8 strategies

best_sharpe = med['Sharpe'].idxmax()
highest_return = med['Ann. return'].idxmax()
lowest_vol = med['Ann. volatility'].idxmin()
smallest_drawdown = med['Max drawdown'].idxmax()   # the drawdown is negative: the "largest" is the least bad
highest_turnover = med['Turnover (annual)'].idxmax()
most_concentrated = med['Effective N'].idxmin()

print('MAIN RESULTS (medians of the 100 experiments, net of 10 bps)')
print(f"- Highest Sharpe: {best_sharpe} ({med.loc[best_sharpe, 'Sharpe']:.2f}) | EW: {med.loc['EW', 'Sharpe']:.2f} | SPY: {medians.loc['SPY (buy & hold)', 'Sharpe']:.2f}")
print(f"- Highest return: {highest_return} ({med.loc[highest_return, 'Ann. return']:.1%}) | EW: {med.loc['EW', 'Ann. return']:.1%}")
print(f"- Lowest volatility: {lowest_vol} ({med.loc[lowest_vol, 'Ann. volatility']:.1%}) | EW: {med.loc['EW', 'Ann. volatility']:.1%}")
print(f"- Smallest maximum drawdown: {smallest_drawdown} ({med.loc[smallest_drawdown, 'Max drawdown']:.1%}) | EW: {med.loc['EW', 'Max drawdown']:.1%}")
print(f"- Highest turnover: {highest_turnover} ({med.loc[highest_turnover, 'Turnover (annual)']:.2f} per year) | EW: {med.loc['EW', 'Turnover (annual)']:.2f}")
print(f"- Most concentrated: {most_concentrated} (effective N = {med.loc[most_concentrated, 'Effective N']:.1f} out of {N_ASSETS})")

print('\n% OF EXPERIMENTS IN WHICH EACH STRATEGY BEATS EW (Sharpe | return | drawdown | volatility)')
for name in beats_ew.index:
    print(f"- {name:20s}: {beats_ew.loc[name, '% beats EW: Sharpe']:4.0f}% | {beats_ew.loc[name, '% beats EW: Ann. return']:4.0f}% | "
          f"{beats_ew.loc[name, '% beats EW: Max drawdown']:4.0f}% | {beats_ew.loc[name, '% beats EW: Ann. volatility']:4.0f}%")

print('\nROBUSTNESS')
changed = ranking_by_cost[ranking_by_cost.nunique(axis=1) > 1].index.tolist()
print(f'- Strategies that change position in the ranking between 0 and 50 bps: {len(changed)} out of {len(ranking_by_cost)} {changed}')
print(f"- Best strategy (median Sharpe) with quarterly rebalancing: {rebalance_comparison['Sharpe (quarterly)'].idxmax()}")
for version in ['12 stocks (main)', '12 stocks (new draw)', 'All eligible stocks']:
    print(f"- Best strategy with '{version}': {subset_comparison[(version, 'Median Sharpe')].idxmax()}")
print(f'- Optimisations that failed (all runs): {len(failures_table)}')
print(f'- Optimisations that needed a second attempt (ftol = 1e-8): {len(retry_log)} '
      f'(of which failed again and used equal weights: {len(failures_table)})')

## 14. Conclusions and recommendation (guide for the report)

> The exact numbers are those in the automatic summary (section 13) and in the tables above. The "≈" values below come from results that
> **do not depend on the risk-free rate** (returns, volatility, drawdown, turnover, concentration). The statements about the **Sharpe ratio**
> and about **Max Sharpe** must be confirmed with your own run (with the real DGS3 series).

**1. Equally weighted is very hard to beat.** No optimised strategy beats EW on Sharpe ratio consistently:
the best ones (ERC, most diversified, inverse vol) are close to 50% of the experiments; ERC and inverse vol have a median Sharpe ratio
practically equal to that of EW, and most diversified is ≈ 0.1 below.
With a new draw of stocks, or with all the stocks (section 10.6), the order among these strategies changes → **the differences between them are not robust**.

**2. Low-risk family (GMV, inverse vol, risk parity).** They reduce volatility relative to EW in ≈ 98–100% of the experiments
and the maximum drawdown in ≈ 65–87%, but they earn less (return above EW in only ≈ 26–27% of the experiments).
Inverse vol and ERC do this with turnover similar to that of EW (≈ 0.3 per year) and well-diversified portfolios (effective N ≈ 11 out of 12).
GMV is more concentrated (effective N ≈ 4) and has the lowest median return.

**3. Strategies that use expected returns (Markowitz λ = 3 and Max Sharpe).** Very concentrated portfolios
(Markowitz: effective N ≈ 1.5 and average maximum weight ≈ 77%), turnover ≈ 8 times that of EW, more volatility and larger drawdowns.
Markowitz has the highest median return, but also the largest dispersion (standard deviation of the annual return ≈ 0.68 against ≈ 0.22 for EW)
and it only has a smaller drawdown than EW in ≈ 9% of the experiments. This is the effect of **estimation error**: 1-year means
point to the stocks that rose the most, which do not always keep rising. A higher λ (5) reduces risk, but does not solve the concentration.

**4. Most diversified and Max decorrelation** sit halfway: more return than the low-risk family,
but turnover ≈ 3 times that of EW and moderate concentration (effective N ≈ 6–7).

**5. Transaction costs.** With 10 bps the annual cost is small for all of them (median ≈ 0.3% per year for Markowitz, the most expensive strategy;
≈ 1% in the worst experiment) and the ranking by Sharpe ratio **practically does not change** between 0 and 50 bps: at most, strategies
with an almost identical median Sharpe ratio, such as EW and ERC, swap places (confirm in section 10.1 and in the summary in section 13). In this universe of very liquid stocks, what separates the strategies is estimation error, not costs.

**6. Quarterly rebalancing** increases turnover by ≈ 30–45% without consistently improving performance →
**semi-annual** rebalancing is enough.

**7. Dependence on the period.** In the years in which SPY fell (12 experiments), EW and the low-risk strategies hardly lost anything
(median return ≈ 0% to +4%), whereas Markowitz was the only strategy with a negative median return (≈ −5%, close to the ≈ −5% of SPY).
Markowitz also gets much worse in the 2021–2026 sub-period. The results vary a lot between experiments — that is why we look at medians
and percentages and not at a single experiment.

**8. SPY.** SPY has the lowest median Sharpe ratio and one of the lowest returns (only GMV earns less), but it has the **smallest median drawdown
of all** (≈ −9%) and a low volatility (≈ 14%; only GMV has less). The difference in return mainly reflects the
**selection bias** of the universe (the 23 stocks are the largest **today**) and not a real advantage of the strategies over the market.

### Suggested recommendation (for the group to decide)

- **Target investor:** a long-term investor with an allocation to US equities who wants to **reduce volatility and drawdowns**
  with low costs and low complexity (for example, a pension fund or a prudent private investor).
- **Recommendation:** **risk parity (ERC)** — or inverse vol, which is even simpler. They have a Sharpe ratio similar to EW,
  lower volatility in practically every experiment, smaller drawdowns in the large majority of them, low turnover and stable weights.
  The cost is a lower return than EW (≈ 5 percentage points per year at the median: ≈ 26% against ≈ 31%).
- For those who want **more return** and accept the volatility of equities, **EW** is the simplest and most robust choice (it does not need to estimate anything).
  Markowitz and Max decorrelation have a higher median return (≈ 40% and ≈ 33% against ≈ 31%), but with more volatility, more dispersion
  across experiments and results that depend on the stocks drawn (section 10.6).
- **We do not recommend** unconstrained Markowitz or Max Sharpe: extreme concentration, a lot of turnover and unstable results.
- **When to reconsider:** (i) if the correlations between stocks change a lot (ERC depends on the covariance matrix);
  (ii) in markets that are very concentrated in a few winning stocks, where EW and ERC hold little weight in those stocks;
  (iii) if the costs or the universe are very different (e.g. illiquid stocks); (iv) if weight limits
  or a *shrinkage* covariance are added (extensions A and B of the brief), which may make Markowitz and Max Sharpe more stable.